In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
import joblib

In [2]:
from sklearn.datasets import load_breast_cancer

breast_cancer = load_breast_cancer()
data_df = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
data_df['target'] = breast_cancer.target

print(f"Dataset shape: {data_df.shape}")
print(f"Features: {data_df.shape[1] - 1}")
print(f"Samples: {data_df.shape[0]}")
print(f"Target classes: {breast_cancer.target_names}")

# Display basic info
data_df.info()

Dataset shape: (569, 31)
Features: 30
Samples: 569
Target classes: ['malignant' 'benign']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    flo

In [3]:
print("Target distribution:")
print(data_df['target'].value_counts())
print(f"Class balance: {data_df['target'].value_counts(normalize=True)}")
print(f"0: {breast_cancer.target_names[0]}, 1: {breast_cancer.target_names[1]}")

Target distribution:
target
1    357
0    212
Name: count, dtype: int64
Class balance: target
1    0.627417
0    0.372583
Name: proportion, dtype: float64
0: malignant, 1: benign


In [4]:
# Prepare features and target
X = data_df.drop('target', axis=1)
y = data_df['target']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")

Training set: (455, 30)
Test set: (114, 30)


In [5]:
# Function to calculate all metrics
def calculate_all_metrics(y_test, y_pred, y_pred_probability):
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC Score': roc_auc_score(y_test, y_pred_probability),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'MCC Score': matthews_corrcoef(y_test, y_pred)
    }
    
    return metrics

In [6]:
#Logistic Regression

# Instantiate model
logisticRegressionModel = LogisticRegression(random_state=42, solver='liblinear')

# Fit the model
logisticRegressionModel.fit(X_train, y_train)

# Make predictions on the test data
y_pred_logistic_regression = logisticRegressionModel.predict(X_test)

# Obtain positive class probability predictions
y_pred_probability_logistic_regression = logisticRegressionModel.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics

metrics = calculate_all_metrics(y_test, y_pred_logistic_regression, y_pred_probability_logistic_regression)

print("--- Logistic Regression Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")

logistic_regression_metrics = {
    'Model': 'Logistic Regression',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nLogistic Regression Metrics stored:")
print(logistic_regression_metrics)

joblib.dump(logisticRegressionModel, f"model/logisticRegressionModel.pkl")

--- Logistic Regression Model Evaluation ---
Accuracy: 0.9561
AUC Score: 0.9957
Precision: 0.9589
Recall: 0.9722
F1 Score: 0.9655
MCC Score: 0.9054

Logistic Regression Metrics stored:
{'Model': 'Logistic Regression', 'Accuracy': 0.956140350877193, 'AUC Score': 0.9957010582010581, 'Precision': 0.958904109589041, 'Recall': 0.9722222222222222, 'F1 Score': 0.9655172413793104, 'MCC': 0.9054466190452621}


['model/logisticRegressionModel.pkl']

In [7]:
# Decision Tree Classifier

# Instantiate model
decisionTreeClassifierModel = DecisionTreeClassifier(random_state=42)

# Fit the model
decisionTreeClassifierModel.fit(X_train, y_train)

# Make predictions on the test data
y_pred_decision_tree_classifier = decisionTreeClassifierModel.predict(X_test)

# Obtain positive class probability predictions
y_pred_probability_decision_tree_classifier = decisionTreeClassifierModel.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_test, y_pred_decision_tree_classifier, y_pred_probability_decision_tree_classifier)


print("--- Decision Tree Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")

decision_tree_metrics = {
    'Model': 'Decision Tree Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nDecision Tree Classifier Metrics stored:")
print(decision_tree_metrics)

joblib.dump(decisionTreeClassifierModel, f"model/decisionTreeClassifierModel.pkl")

--- Decision Tree Classifier Model Evaluation ---
Accuracy: 0.9123
AUC Score: 0.9157
Precision: 0.9559
Recall: 0.9028
F1 Score: 0.9286
MCC Score: 0.8174

Decision Tree Classifier Metrics stored:
{'Model': 'Decision Tree Classifier', 'Accuracy': 0.9122807017543859, 'AUC Score': 0.9156746031746031, 'Precision': 0.9558823529411765, 'Recall': 0.9027777777777778, 'F1 Score': 0.9285714285714286, 'MCC': 0.8174119974927639}


['model/decisionTreeClassifierModel.pkl']

In [8]:
# KNN Classifier

# Instantiate model
knnModel = KNeighborsClassifier(n_neighbors=5)

# Fit the model
knnModel.fit(X_train, y_train)

# Make predictions on the test data
y_pred_knn = knnModel.predict(X_test)

# Obtain positive class probability predictions
y_pred_probability_knn = knnModel.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_test, y_pred_knn, y_pred_probability_knn)

print("--- K-Nearest Neighbor Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")

knn_metrics = {
    'Model': 'K-Nearest Neighbor Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nK-Nearest Neighbor Classifier Metrics stored:")
print(knn_metrics)

joblib.dump(knnModel, f"model/knnModel.pkl")

--- K-Nearest Neighbor Classifier Model Evaluation ---
Accuracy: 0.9123
AUC Score: 0.9559
Precision: 0.9429
Recall: 0.9167
F1 Score: 0.9296
MCC Score: 0.8139

K-Nearest Neighbor Classifier Metrics stored:
{'Model': 'K-Nearest Neighbor Classifier', 'Accuracy': 0.9122807017543859, 'AUC Score': 0.9558531746031746, 'Precision': 0.9428571428571428, 'Recall': 0.9166666666666666, 'F1 Score': 0.9295774647887324, 'MCC': 0.8139267835041308}


['model/knnModel.pkl']

In [9]:
# Naive Bayes Classifier

# Instantiate model
naiveBayesModel = GaussianNB()

# Fit the model
naiveBayesModel.fit(X_train, y_train)

# Make predictions on the test data
y_pred_naive_bayes = naiveBayesModel.predict(X_test)

# Obtain positive class probability predictions
y_pred_probability_naive_bayes = naiveBayesModel.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_test, y_pred_naive_bayes, y_pred_probability_naive_bayes)


print("--- Naive Bayes Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")


naive_bayes_metrics = {
    'Model': 'Naive Bayes Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nNaive Bayes Classifier Metrics stored:")
print(naive_bayes_metrics)

joblib.dump(naiveBayesModel, f"model/naiveBayesModel.pkl")

--- Naive Bayes Classifier Model Evaluation ---
Accuracy: 0.9386
AUC Score: 0.9878
Precision: 0.9452
Recall: 0.9583
F1 Score: 0.9517
MCC Score: 0.8676

Naive Bayes Classifier Metrics stored:
{'Model': 'Naive Bayes Classifier', 'Accuracy': 0.9385964912280702, 'AUC Score': 0.9877645502645502, 'Precision': 0.9452054794520548, 'Recall': 0.9583333333333334, 'F1 Score': 0.9517241379310345, 'MCC': 0.8675534786006366}


['model/naiveBayesModel.pkl']

In [10]:
# Random Forest Classifier

# Instantiate model
randomForestmodel = RandomForestClassifier(random_state=42)

# Fit the model
randomForestmodel.fit(X_train, y_train)

# Make predictions on the test data
y_pred_random_forest = randomForestmodel.predict(X_test)

# Obtain positive class probability predictions
y_pred_probability_random_forest = randomForestmodel.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_test, y_pred_random_forest, y_pred_probability_random_forest)


print("--- Random Forest Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")


random_forest_metrics = {
    'Model': 'Random Forest Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nRandom Forest Classifier Metrics stored:")
print(random_forest_metrics)

joblib.dump(randomForestmodel, f"model/randomForestmodel.pkl")

--- Random Forest Classifier Model Evaluation ---
Accuracy: 0.9561
AUC Score: 0.9937
Precision: 0.9589
Recall: 0.9722
F1 Score: 0.9655
MCC Score: 0.9054

Random Forest Classifier Metrics stored:
{'Model': 'Random Forest Classifier', 'Accuracy': 0.956140350877193, 'AUC Score': 0.9937169312169312, 'Precision': 0.958904109589041, 'Recall': 0.9722222222222222, 'F1 Score': 0.9655172413793104, 'MCC': 0.9054466190452621}


['model/randomForestmodel.pkl']

In [11]:
# XGBoost Classifier

# Instantiate model
xgBoostModel = XGBClassifier(random_state=42, eval_metric='logloss') # use_label_encoder is removed as it is deprecated and no longer needed

# Fit the model
xgBoostModel.fit(X_train, y_train)

# Make predictions on the test data
y_pred_xgboost = xgBoostModel.predict(X_test)

# Obtain positive class probability predictions
y_pred_probability_xgboost = xgBoostModel.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_test, y_pred_xgboost, y_pred_probability_xgboost)


print("--- XGBoost Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")


xgboost_metrics = {
    'Model': 'XGBoost Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nXGBoost Classifier Metrics stored:")
print(xgboost_metrics)

joblib.dump(xgBoostModel, f"model/xgBoostModel.pkl")

--- XGBoost Classifier Model Evaluation ---
Accuracy: 0.9561
AUC Score: 0.9901
Precision: 0.9467
Recall: 0.9861
F1 Score: 0.9660
MCC Score: 0.9058

XGBoost Classifier Metrics stored:
{'Model': 'XGBoost Classifier', 'Accuracy': 0.956140350877193, 'AUC Score': 0.9900793650793651, 'Precision': 0.9466666666666667, 'Recall': 0.9861111111111112, 'F1 Score': 0.9659863945578231, 'MCC': 0.9058238738943076}


['model/xgBoostModel.pkl']

In [13]:
# Comparison

# Collect all metrics dictionaries into a list
all_metrics = [
    logistic_regression_metrics,
    decision_tree_metrics,
    knn_metrics,
    naive_bayes_metrics,
    random_forest_metrics,
    xgboost_metrics
]

# Create a DataFrame from the list of dictionaries
metrics_df = pd.DataFrame(all_metrics)

# Display the comparison table, formatting numerical columns for better readability
# Exclude the 'Model' column from formatting to keep it as string
formatted_metrics_df = metrics_df.copy()
for col in formatted_metrics_df.columns:
    if col != 'Model':
        formatted_metrics_df[col] = formatted_metrics_df[col].apply(lambda x: f"{x:.4f}")

print("\n--- Model Performance Comparison Table ---")
print(formatted_metrics_df.to_string())


--- Model Performance Comparison Table ---
                           Model Accuracy AUC Score Precision  Recall F1 Score     MCC
0            Logistic Regression   0.9561    0.9957    0.9589  0.9722   0.9655  0.9054
1       Decision Tree Classifier   0.9123    0.9157    0.9559  0.9028   0.9286  0.8174
2  K-Nearest Neighbor Classifier   0.9123    0.9559    0.9429  0.9167   0.9296  0.8139
3         Naive Bayes Classifier   0.9386    0.9878    0.9452  0.9583   0.9517  0.8676
4       Random Forest Classifier   0.9561    0.9937    0.9589  0.9722   0.9655  0.9054
5             XGBoost Classifier   0.9561    0.9901    0.9467  0.9861   0.9660  0.9058
